# 01. 피처 스토어로 학습과 서빙에 같은 피처 주기

책 4.2절. 시나리오 7의 API는 원본 컬럼 8개를 one-hot 인코딩한 컬럼 90개를 받습니다. 이 노트북은 인코딩한 피처를 Feast에 한 번 등록하고,
학습할 때는 오프라인 스토어(object storage의 parquet)에서, 서빙할 때는 온라인 스토어(Redis)에서 같은 값을 꺼냅니다. API를 부르는 쪽은 `person_id`만 압니다.

커널은 **Python (feast)** 를 고릅니다.

In [ ]:
import io
import sys
from datetime import datetime, timezone

import boto3
import pandas as pd
import requests
from feast import FeatureStore

sys.path += ["feature_repo", "../mlflow"]
from features import income_features, person
from income import load_income_data

SERVING_URL = "http://income-classifier-predictor.kubeflow-user-example-com.svc.cluster.local/invocations"

피처 테이블을 만듭니다. 시나리오 5의 학습 코드와 같은 one-hot 인코딩에 사람을 구분할 `person_id`와 피처를 계산한 시각 `event_timestamp`를 더합니다.

In [ ]:
df = load_income_data("../mlflow/income_data.csv")
features = pd.get_dummies(df.drop("Target", axis=1), drop_first=True, dtype=float)
COLUMNS = features.columns.to_list()
features.insert(0, "person_id", range(len(features)))
features["event_timestamp"] = datetime.now(timezone.utc)

buffer = io.BytesIO()
features.to_parquet(buffer, index=False)
boto3.client("s3").put_object(Bucket="feature-data-sets", Key="income/features.parquet", Body=buffer.getvalue())
features.shape

피처 정의(`feature_repo/features.py`)를 레지스트리에 등록하고, 오프라인 스토어의 최신 값을 Redis로 옮깁니다(materialize).

In [ ]:
store = FeatureStore(repo_path="feature_repo")
store.apply([person, income_features])
store.materialize_incremental(end_date=datetime.now(timezone.utc))
print("feature view:", income_features.name, "features:", len(store.get_feature_view("income_features").features))

서빙 쪽: `person_id`만으로 Redis에서 피처 90개를 꺼내 KServe에 보냅니다. 인코딩 코드는 이 셀에 없습니다.

In [ ]:
def predict(person_ids: list[int]) -> list[float]:
  """온라인 스토어에서 피처를 꺼내 시나리오 7의 InferenceService로 예측한다."""
  online = store.get_online_features(
    features=[f"income_features:{c}" for c in COLUMNS],
    entity_rows=[{"person_id": i} for i in person_ids],
  ).to_df()
  body = {"dataframe_split": {"columns": COLUMNS, "data": online[COLUMNS].values.tolist()}}
  return requests.post(SERVING_URL, json=body, timeout=30).json()["predictions"]

print(predict([0, 1, 2]))

학습 쪽: 같은 피처를 오프라인 스토어에서 시점 기준으로 꺼냅니다(point-in-time join). 서빙이 받은 값과 학습이 받는 값이 같은지 확인합니다.

In [ ]:
entity_df = pd.DataFrame({"person_id": [0, 1, 2], "event_timestamp": datetime.now(timezone.utc)})
offline = store.get_historical_features(entity_df=entity_df, features=[f"income_features:{c}" for c in COLUMNS]).to_df()
online = store.get_online_features(features=[f"income_features:{c}" for c in COLUMNS],
                                   entity_rows=[{"person_id": i} for i in [0, 1, 2]]).to_df()
same = (offline.sort_values("person_id")[COLUMNS].values == online[COLUMNS].values).all()
print("offline == online:", same)